In [0]:
%pip install pyspark nltk pandas transformers torch


In [0]:
import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession, SQLContext, functions as F
from pyspark.sql.functions import *


In [0]:
#Enabling SparkSession
spark = SparkSession \
.builder \
.appName("ABC") \
.config("spark.driver.memory", "15g") \
.config("spark.mongodb.read.connection.uri", "MongoDB Connection String") \
.config("spark.mongodb.write.connection.uri", "MongoDB Connection String") \
.config('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector:10.0.2') \
.getOrCreate()

In [0]:
%python
#Spark Connection to MongoDB Atlas (Read)
mongo_uri = "MongoDB Connection String"

df = spark.read \
.format("com.mongodb.spark.sql.DefaultSource") \
.option("uri", mongo_uri) \
.option("database", "NVIDIA") \
.option("collection", "Stock_Sentiment") \
.load()

display(df)

In [0]:
#Function Define

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)                     # Remove HTML
    text = re.sub(r'http\S+|www\S+', '', text)            # Remove URLs
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)            # Remove special chars
    #words = word_tokenize(text)
    #words = [word for word in words if word not in stop_words]
    return text



In [0]:
#Running Clean Text

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


def full_clean(text):
    cleaned = clean_text(text)
    return cleaned

full_clean_udf = udf(full_clean, StringType())

df_cleaned = df.withColumn("clean_body", full_clean_udf(df["text"]))

display(df_cleaned)

In [0]:
#Remove Empty Rows and dup IDs
from pyspark.sql.functions import col, split

# Remove rows where 'text' column is empty
df_cleaned = df_cleaned.filter(col('text').isNotNull() & (col('text') != ''))

# Remove rows with repeating 'id'
df_cleaned_dup = df_cleaned.dropDuplicates(['id'])

# Split 'clean_body' into multiple rows if token count exceeds 500
df_cleaned_dup = df_cleaned_dup.withColumn("clean_body_split", split(col("clean_body"), " "))

display(df_cleaned_dup)

In [0]:
#Change Dataframe into Pandas
pandas_df = df_cleaned_dup.select("id", "date", "clean_body").toPandas()

display(pandas_df)

In [0]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

pandas_df["num_tokens"] = pandas_df["clean_body"].apply(lambda x: len(tokenizer.tokenize(x)))
pandas_df_filtered = pandas_df[pandas_df["num_tokens"] <= 490].copy()


In [0]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="yiyanghkust/finbert-tone",
    tokenizer="yiyanghkust/finbert-tone"
)

def truncate_text(text, max_length=512):
    tokens = sentiment_model.tokenizer.tokenize(text)
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
    return sentiment_model.tokenizer.convert_tokens_to_string(tokens)

pandas_df_filtered["sentiment_result"] = pandas_df_filtered["clean_body"].apply(
    lambda x: sentiment_model(truncate_text(x[ :512]))[0]
)


In [0]:
def truncate_text(text, max_length=512):
    tokens = sentiment_model.tokenizer.tokenize(text)
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
    print(tokens)  #  This will print the tokenized version
    return sentiment_model.tokenizer.convert_tokens_to_string(tokens)

pandas_df_filtered["sentiment_result"] = pandas_df_filtered["clean_body"].apply(
    lambda x: sentiment_model(truncate_text(x[:512]))[0]
)


Message #bigdata-code-and-pics


In [0]:
#Split the sentiment result into label and score
pandas_df_filtered["sentiment_label"] = pandas_df_filtered["sentiment_result"].apply(lambda x: x["label"])
pandas_df_filtered["sentiment_score"] = pandas_df_filtered["sentiment_result"].apply(lambda x: x["score"])


In [0]:
import pandas as pd

# Convert the pandas_df_filtered to a Spark DataFrame
spark_df_filtered = spark.createDataFrame(pandas_df_filtered)

# Display the new Spark DataFrame
display(spark_df_filtered)

In [0]:
#Settings for Write (MongoDB)
write_config = {
    "uri": "MongoDB Connection String",
    "database": "NVIDIA",
    "collection": "Stock_Sentiment_Refined",
    "writeConcern.w": "majority"
}

In [0]:
#Write Dataframe back to MongoDB
spark_df_filtered.write.format("mongo") \
    .mode("append") \
    .options(**write_config) \
    .save()